Python execute their code in synchronous way by following top to down approach ( by default its blocking). call next line of code afterr executing current one

- when we  need to work in asynchronous ( non-blocking) you need to use asyncio


# synchronous ( blocking)

In [10]:
# 1- example
import time

def timelog (number,delay) :
    
    for i in range(number):
     print(f"Count: {i + 1} of {number}")
     time.sleep(delay)

    
timelog(5,15)
# 15 x 5 (second)  => execute 
print("hello")

Count: 1 of 5
Count: 2 of 5
Count: 3 of 5
Count: 4 of 5
Count: 5 of 5
hello


In [ ]:
#2 Example

with open("data_4_python.txt","r" , encoding="utf-8") as file : 
    content  = file.read()
    print(content)

India is a vast and multifaceted civilization defined by its immense geographical breadth, millenniums-spanning history, profound cultural plurality, and rapid modern transformation. Stretching from the snow-capped Himalayan ridges in the north to the tropical waters of Kanyakumari at its southern tip, the subcontinent encompasses virtually every major biome on Earth, including the arid expanse of the Thar Desert, the fertile alluvial plains of the Indo-Gangetic basin, the dense rainforests of the Western Ghats, and extensive coastlines along the Arabian Sea and the Bay of Bengal.



# asynchronous ( non-blocking)

In [11]:
#Example 1
import asyncio

async def timelog(name, number, delay):
    for i in range(number):
        print(f"[{name}] {i + 1} of {number}", flush=True)
        await asyncio.sleep(delay)

print("you called me before async function call")

task1 = asyncio.create_task(timelog("Task A", 3, 1))
task2 = asyncio.create_task(timelog("Task B", 3, 0.5))

print("you called me after async function call (running concurrently now!)")


await asyncio.gather(task1, task2)
print("All background tasks completed!")

you called me before async function call
you called me after async function call (running concurrently now!)
[Task A] 1 of 3
[Task B] 1 of 3
[Task B] 2 of 3
[Task A] 2 of 3
[Task B] 3 of 3
[Task A] 3 of 3
All background tasks completed!


In [23]:
# 2 Example
import asyncio

def _read_file(file_location):
    with open(file_location, "r", encoding="utf-8") as file:
        return file.read()

async def async_file_read(file_location):
    content = await asyncio.to_thread(_read_file, file_location)
    return content

read = asyncio.create_task(async_file_read("data_4_python.txt"))
print(f"Task status: {read}")
print("called before read.")
await asyncio.gather(read)

Task status: <Task pending name='Task-932' coro=<async_file_read() running at C:\Users\anand\AppData\Local\Temp\ipykernel_5264\2017070216.py:8>>
called before read.


["India is a vast and multifaceted civilization defined by its immense geographical breadth, millenniums-spanning history, profound cultural plurality, and rapid modern transformation. Stretching from the snow-capped Himalayan ridges in the north to the tropical waters of Kanyakumari at its southern tip, the subcontinent encompasses virtually every major biome on Earth, including the arid expanse of the Thar Desert, the fertile alluvial plains of the Indo-Gangetic basin, the dense rainforests of the Western Ghats, and extensive coastlines along the Arabian Sea and the Bay of Bengal.\n\nCivilizational Roots and History\n\nHuman civilization in India traces back to the ancient Indus Valley Civilization (mature period c. 2600–1900 BCE), which pioneered urban planning, standardized brick weights, and advanced drainage systems in sites such as Harappa, Mohenjo-daro, and Dholavira. The subsequent Vedic period laid foundational philosophical and literary traditions through the composition of 

# Generator function

generator is a function that produces a sequence of values on demand (one at a time) instead of computing everything upfront and storing it in memory. It pauses its state at yield and resumes only when asked for the next item.

The Real-World Problem: Reading Massive Files
Imagine you have a 10 GB server log file, and your machine only has 8 GB of RAM.

The Problem (Using a Regular List): If you try to read all lines into a standard list at once (file.readlines()), your program attempts to allocate 10 GB into RAM, crashing immediately with MemoryError.

The Generator Solution: A generator streams one line at a time into memory, processes it, discards it, and moves to the next. Memory usage stays constant at just a few kilobytes.

Standard List Approach (Crash):
[10 GB File] ──Load All──> [8 GB RAM] ──> 💥 MemoryError (Out of Memory)

Generator Approach (Success):
[10 GB File] ──1 Line at a time──> [RAM (KB)] ──Process & Discard──> Repeat

In [4]:
import os

sample_logs = """[2026-08-16 01:00:01] INFO  path=/home status=200 device=Desktop
[2026-08-16 01:00:02] ERROR path=/pricing status=404 device=Mobile
[2026-08-16 01:00:03] INFO  path=/about status=200 device=Mobile
[2026-08-16 01:00:04] ERROR path=/contact status=500 device=Desktop
[2026-08-16 01:00:05] ERROR path=/old-page status=404 device=Mobile
[2026-08-16 01:00:06] ERROR path=/missing status=404 device=Desktop
[2026-08-16 01:00:07] ERROR path=/api/v1/user status=404 device=Mobile
"""

with open("server.log", "w") as f:
    f.write(sample_logs.strip())

def stream_large_log(file_path):
    """Generator: Reads one line at a time from disk into RAM."""
    with open(file_path, "r") as file:
        for line in file:
            yield line.strip()


def filter_status_code(log_stream, status_code="404"):
    """Generator: Filters lines on the fly without building an intermediate list."""
    for line in log_stream:
        if f"status={status_code}" in line:
            yield line

raw_logs = stream_large_log("server.log")
failed_requests = filter_status_code(raw_logs, status_code="404")

mobile_404_count = 0
print("Matching 404 Log Line")
for error_line in failed_requests:
    print(f"Found: {error_line}")
    if "device=Mobile" in error_line:
        mobile_404_count += 1

print("\nSummary")
print(f"Total 404 errors on Mobile: {mobile_404_count}")


os.remove("server.log")

Matching 404 Log Line
Found: [2026-08-16 01:00:02] ERROR path=/pricing status=404 device=Mobile
Found: [2026-08-16 01:00:05] ERROR path=/old-page status=404 device=Mobile
Found: [2026-08-16 01:00:06] ERROR path=/missing status=404 device=Desktop
Found: [2026-08-16 01:00:07] ERROR path=/api/v1/user status=404 device=Mobile

Summary
Total 404 errors on Mobile: 3


# Dataclasses

In GenAI, a dataclass is a type-safe data contract that structures messy LLM outputs, prompt parameters, and vector retrieval chunks into predictable, validated objects. It replaces error-prone dictionary lookups (response["data"]["score"]) with strict schema definitions, catching missing keys, invalid types, and malformed JSON before bad data breaks the AI pipeline.

End-to-End GenAI Code Demonstration
The following production script demonstrates how dataclasses manage parsing, schema validation, and error recovery when dealing with raw LLM outputs (including simulated malformed responses).

In [6]:
import json
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any


@dataclass(frozen=True)
class PromptConfig:
    """Immutable configuration for the LLM request."""
    model: str
    temperature: float = 0.2
    max_tokens: int = 512

@dataclass
class RetrievedChunk:
    """Represents a vector search result from a knowledge base."""
    text: str
    source_url: str
    similarity_score: float
    relevance_grade: str = field(init=False)

    def __post_init__(self):
        # Validate score bounds and compute derived classification
        if not (0.0 <= self.similarity_score <= 1.0):
            raise ValueError(f"Invalid similarity score: {self.similarity_score}")
        self.relevance_grade = "HIGH" if self.similarity_score >= 0.8 else "LOW"

@dataclass
class AIEntityExtractionResponse:
    """Expected structured schema returned by the LLM."""
    query: str
    entities: List[str]
    confidence: float
    reasoning: str
    chunks_used: List[RetrievedChunk] = field(default_factory=list)

    @classmethod
    def from_llm_json(cls, raw_json_str: str, chunks: List[RetrievedChunk]) -> "AIEntityExtractionResponse":
        """Parses and validates raw LLM JSON text into a typed dataclass object."""
        try:
            data: Dict[str, Any] = json.loads(raw_json_str)
        except json.JSONDecodeError as e:
            raise ValueError(f"LLM produced invalid JSON: {e}") from e

        required_fields = {"query", "entities", "confidence", "reasoning"}
        missing = required_fields - data.keys()
        if missing:
            raise KeyError(f"LLM JSON missing required fields: {missing}")

        return cls(
            query=data["query"],
            entities=list(data["entities"]),
            confidence=float(data["confidence"]),
            reasoning=data["reasoning"],
            chunks_used=chunks
        )


# ==========================================
# 2. Pipeline Execution (Success & Failure)
# ==========================================

def run_genai_pipeline(raw_llm_output: str, source_chunks: List[RetrievedChunk]):
    print("--------------------------------------------------")
    print(f"Raw Input: {raw_llm_output}")
    try:
        # Attempt to parse raw LLM output into our strict dataclass contract
        structured_response = AIEntityExtractionResponse.from_llm_json(
            raw_json_str=raw_llm_output,
            chunks=source_chunks
        )
        print("\nSUCCESS: Parsed into Dataclass successfully!")
        print(f"Entities Found:  {structured_response.entities}")
        print(f"Model Confidence: {structured_response.confidence * 100:.1f}%")
        print(f"Top Chunk Grade: {structured_response.chunks_used[0].relevance_grade}")
        print(f"Serialized Dict: {asdict(structured_response)}")

    except (ValueError, KeyError) as err:
        print("\nPIPELINE RECOVERY: Caught malformed LLM response!")
        print(f"Error Type:   {type(err).__name__}")
        print(f"Root Cause:   {err}")
        print("Fallback:     Triggering schema self-correction / retrying prompt...")


# Mock Vector DB Chunk
sample_chunks = [
    RetrievedChunk(
        text="Python dataclasses simplify state storage in AI pipelines.",
        source_url="https://docs.python.org/3/library/dataclasses.html",
        similarity_score=0.92
    )
]

# Scenario A: Valid LLM Output
valid_llm_json = """{
    "query": "Extract tech names",
    "entities": ["Python", "dataclasses"],
    "confidence": 0.96,
    "reasoning": "Both terms explicitly refer to programming technologies."
}"""

# Scenario B: Failed LLM Output (Broken JSON Syntax)
broken_json_llm = """{
    "query": "Extract tech names",
    "entities": ["Python", "dataclasses",
    "confidence": 0.96
}"""

# Scenario C: Failed LLM Output (Missing Required Fields / Hallucinated Keys)
missing_keys_llm = """{
    "search_term": "Extract tech names",
    "extracted_items": ["Python"],
    "notes": "Missing confidence score"
}"""

print("=== SCENARIO 1: VALID LLM RESPONSE ===")
run_genai_pipeline(valid_llm_json, sample_chunks)

print("\n=== SCENARIO 2: SYNTAX ERROR FROM LLM ===")
run_genai_pipeline(broken_json_llm, sample_chunks)

print("\n=== SCENARIO 3: MISSING SCHEMA FIELDS FROM LLM ===")
run_genai_pipeline(missing_keys_llm, sample_chunks)

=== SCENARIO 1: VALID LLM RESPONSE ===
--------------------------------------------------
Raw Input: {
    "query": "Extract tech names",
    "entities": ["Python", "dataclasses"],
    "confidence": 0.96,
    "reasoning": "Both terms explicitly refer to programming technologies."
}

SUCCESS: Parsed into Dataclass successfully!
Entities Found:  ['Python', 'dataclasses']
Model Confidence: 96.0%
Top Chunk Grade: HIGH
Serialized Dict: {'query': 'Extract tech names', 'entities': ['Python', 'dataclasses'], 'confidence': 0.96, 'reasoning': 'Both terms explicitly refer to programming technologies.', 'chunks_used': [{'text': 'Python dataclasses simplify state storage in AI pipelines.', 'source_url': 'https://docs.python.org/3/library/dataclasses.html', 'similarity_score': 0.92, 'relevance_grade': 'HIGH'}]}

=== SCENARIO 2: SYNTAX ERROR FROM LLM ===
--------------------------------------------------
Raw Input: {
    "query": "Extract tech names",
    "entities": ["Python", "dataclasses",
    "co

# Pydantics v2 models

pydantic is a data validation, parsing, and serialization library for Python. It uses standard Python type annotations to define data schemas and strictly enforces type safety, constraints, and coercion at runtime.

- Runtime Validation & Type Coercion: Unlike      standard type hints or dataclasses, Pydantic inspects incoming data at runtime, attempts to convert compatible types (e.g., "42" to 42), and raises detailed error messages when the data is invalid.

- High Performance: In version 2, Pydantic's core engine was rewritten in Rust (pydantic-core), making data validation and JSON parsing significantly faster.

- JSON Schema Generation: Automatically generates OpenAPI- and JSON-compliant schemas from Python models, making it the standard schema engine for FastAPI, LLM tool-calling/structured outputs, and data ingestion pipelines.
- Serialization & Settings: Easily serializes models to and from dictionaries or JSON strings, and manages application environment variables through pydantic-settings

In [3]:
from datetime import datetime
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError


class User(BaseModel):
    id: int
    name: str = Field(min_length=2, max_length=50)
    email: str
    age: Optional[int] = Field(default=None, ge=18, le=120)  # Constraint: 18 <= age <= 120
    tags: List[str] = []
    signup_ts: datetime = Field(default_factory=datetime.utcnow)


raw_data = {
    "id": "101",                    
    "name": "Alex",
    "email": "alex@example.com",
    "age": "25",                     
    "tags": ["developer", "admin"],
}

user = User(**raw_data)
print("Validated User Object:", user)
print("User ID type:", type(user.id))  


print("\nExported JSON:", user.model_dump_json(indent=2))


invalid_data = {
    "id": "abc",                     
    "name": "A",                      
    "email": "not-an-email",          
    "age": 16                         
}

try:
    User(**invalid_data)
except ValidationError as e:
    print("\nValidation Errors Caught:")
    for error in e.errors():
        print(f"- Field '{error['loc'][0]}': {error['msg']}")

Validated User Object: id=101 name='Alex' email='alex@example.com' age=25 tags=['developer', 'admin'] signup_ts=datetime.datetime(2026, 8, 16, 20, 10, 53, 363439)
User ID type: <class 'int'>

Exported JSON: {
  "id": 101,
  "name": "Alex",
  "email": "alex@example.com",
  "age": 25,
  "tags": [
    "developer",
    "admin"
  ],
  "signup_ts": "2026-08-16T20:10:53.363439"
}

Validation Errors Caught:
- Field 'id': Input should be a valid integer, unable to parse string as an integer
- Field 'name': String should have at least 2 characters
- Field 'age': Input should be greater than or equal to 18


Note : pydantic works same as like  zod library that are we used in moslty in express.js for taking input from the user for process the validation at runtime or coerce the correct data typs as per schema demand. 